# Решения: fit, predict_proba и порог

**Для преподавателя.** Полный эталон к `lesson.ipynb` и `homework.ipynb`; ученикам до сдачи не показывать.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def find_bank_csv() -> Path:
    for path in (Path("bank_marketing_slim.csv"), Path("../../data/bank_marketing_slim.csv")):
        if path.exists():
            return path.resolve()
    return "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_10_churn_logreg/data/bank_marketing_slim.csv"


CSV_PATH = find_bank_csv()
df = pd.read_csv(CSV_PATH)
target = df["y"].eq("yes").astype(int)
assert len(df) > 0 and set(target.unique()) == {0, 1}
assert "duration" in df.columns  # колонка видна только для разбора утечки
print(f"Строк: {len(df)}; доля yes: {target.mean():.3f}")

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split


## Урок. 1–3. Признаки и split

In [ ]:
FEATURE_COLUMNS = [column for column in df.columns if column not in {"y", "duration"}]
assert "duration" not in FEATURE_COLUMNS, "LEAKAGE: duration известна только после звонка"
assert "y" not in FEATURE_COLUMNS
X = pd.get_dummies(df[FEATURE_COLUMNS], drop_first=True)
assert "duration" not in X.columns
X_train, X_test, y_train, y_test = train_test_split(
    X, target, test_size=0.25, random_state=61, stratify=target
)
model = LogisticRegression(max_iter=1000, solver="liblinear", class_weight="balanced")
model.fit(X_train, y_train)
proba_test = model.predict_proba(X_test)[:, 1]
assert abs(float(y_train.mean()) - float(y_test.mean())) < 0.05


## Урок. 4–5. Fit и вероятности

In [ ]:
assert hasattr(model, 'coef_')
assert len(proba_test) == len(y_test) and np.std(proba_test) > 0


## Урок. 6. Контракт метрик

In [ ]:
def score_at_threshold(y_true, proba, threshold):
    pred = (np.asarray(proba) >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "selected": int(pred.sum()),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
    }
row_05 = score_at_threshold(y_test, proba_test, 0.5)
assert set(row_05) == {'threshold', 'selected', 'precision', 'recall', 'f1'}


## Урок. 7. Сетка порогов

In [ ]:
thresholds = np.arange(0.10, 0.91, 0.05)
threshold_table = pd.DataFrame([score_at_threshold(y_test, proba_test, t) for t in thresholds])
assert threshold_table['selected'].is_monotonic_decreasing


## Урок. 8. Ограничение recall

In [ ]:
eligible = threshold_table[threshold_table['recall'].ge(0.70)]
chosen_row = eligible.sort_values(['precision', 'threshold'], ascending=False).iloc[0]
assert chosen_row['recall'] >= 0.70


## Урок. 9. Рекомендация

In [ ]:
THRESHOLD_NOTE = (f"Порог {chosen_row.threshold:.2f} сохраняет recall={chosen_row.recall:.3f} и среди допустимых строк даёт "
f"precision={chosen_row.precision:.3f}. Это правило уменьшает пропуски отклика, но число звонков равно {int(chosen_row.selected)}. "
"Вывод относится к одной test-выборке; перед кампанией порог нужно перепроверить на новом периоде и при реальном бюджете.")
assert len(THRESHOLD_NOTE) >= 220


## ДЗ. A1. Воспроизводимый pipeline

In [ ]:
assert 'duration' not in FEATURE_COLUMNS and 'duration' not in X.columns
assert len(proba_test) == len(y_test)


## ДЗ. A2. Плотная сетка

In [ ]:
thresholds = np.arange(0.05, 0.96, 0.025)
rows = [score_at_threshold(y_test, proba_test, t) for t in thresholds]
table = pd.DataFrame(rows)
assert len(table) == len(thresholds)


## ДЗ. A3. Максимум F1

In [ ]:
best_f1_row = table.loc[table['f1'].idxmax()]
assert best_f1_row['f1'] == table['f1'].max()


## ДЗ. Challenge. Бюджет

In [ ]:
budget = max(1, int(np.ceil(0.15 * len(y_test))))
budget_rows = table[table['selected'].le(budget)]
budget_choice = budget_rows.sort_values(['recall', 'precision'], ascending=False).iloc[0]
assert budget_choice['selected'] <= budget


## ДЗ. Challenge. Два правила

In [ ]:
DECISION_NOTE = (
    f"Максимум F1 выбирает порог {best_f1_row.threshold:.3f} и балансирует precision и recall без явной цены звонка. "
    f"Бюджетное правило допускает не более {budget} клиентов и выбирает порог {budget_choice.threshold:.3f}; "
    "оно оптимизирует recall внутри ограничения. Если бюджет жёсткий, второе правило честнее бизнес-задаче. "
    "Если стоимость ошибок симметрична, F1 удобнее как сводный критерий. Оба порога оценены только на test."
)
assert len(DECISION_NOTE) >= 260
